# DIM_MENU_ITEM

SCD1 menu-item-dimensie. Selecteert per `menu_item_id` exact één rij — de laatste in `INTEGRATION.DWH_ORDER_DETAIL` op `WA_FROMDATE` — zodat verwijderde entiteiten zichtbaar blijven (zie ADR-0012). `MK_MENU_ITEM` is de root-surrogate `WKR_ORDER_DETAIL` (stabiel over updates/deletes, zie ADR-0010 + ADR-0012). `MA_CREATEDATE` / `MA_CHANGEDATE` / `MA_ISDEL` volgen de formules uit ADR-0012.

Georkestreerd door `views/07_apply_views.ipynb`.

In [ ]:
%sql
CREATE WIDGET TEXT catalog DEFAULT "DEMO";

CREATE OR REPLACE VIEW ${catalog}.DATAMART.DIM_MENU_ITEM AS
WITH ranked AS (
  SELECT
    menu_item_id,
    WKR_ORDER_DETAIL,
    WA_FROMDATE,
    WA_UNTODATE,
    WA_CRUDDTS,
    WA_CRUD,
    ROW_NUMBER() OVER (PARTITION BY menu_item_id ORDER BY WA_FROMDATE DESC) AS rn,
    MIN(WA_CRUDDTS) OVER (PARTITION BY menu_item_id) AS MA_CREATEDATE,
    MAX(CASE WHEN WA_CRUD <> 'C' THEN WA_CRUDDTS END) OVER (PARTITION BY menu_item_id) AS MA_CHANGEDATE
  FROM ${catalog}.INTEGRATION.DWH_ORDER_DETAIL
)
SELECT
  WKR_ORDER_DETAIL AS MK_MENU_ITEM,
  menu_item_id,
  MA_CREATEDATE,
  MA_CHANGEDATE,
  CASE WHEN WA_UNTODATE <> TIMESTAMP '9999-12-31 00:00:00' THEN 1 ELSE 0 END AS MA_ISDEL
FROM ranked
WHERE rn = 1;